# NoSQL & Analytics

Relational engines handle the transactional core of most workloads, but they aren't the right shape for every kind of data. Globally-distributed document state, multi-petabyte data lakes, telemetry streams, graph traversals — each one has its own engine on Azure. This notebook covers them: **Cosmos DB** for operational NoSQL at planet scale, **Synapse Analytics** and **Azure Databricks** for the analytics tier, **Data Factory** for the orchestration glue, and the supporting characters — Table Storage, HDInsight, Stream Analytics, and Microsoft Fabric — for context.

The pattern most large workloads end up with: Cosmos DB or Azure SQL on the operational side, change feed or CDC pumping into ADLS Gen2, Databricks or Synapse Spark crunching the lake, and a serverless SQL surface or Power BI reading the results. Knowing where each engine fits keeps you from forcing the wrong tool on a workload it can't actually handle.

## Azure Cosmos DB — the planet-scale NoSQL

**Cosmos DB** is Microsoft's globally-distributed, multi-model NoSQL database. The same underlying engine speaks five wire-compatible APIs, and you pick one at create time:

- **NoSQL (Core)** — Cosmos's native JSON document API. Most features land here first; pick it for greenfield.
- **MongoDB** — wire-compatible with MongoDB drivers; not all Mongo features, but enough for most apps. Use it to lift-and-shift Mongo workloads.
- **Cassandra** — CQL wire compatibility. Same migration story for Cassandra.
- **Gremlin** — graph traversals over a property graph. For social graphs, knowledge graphs, fraud rings.
- **Table** — wire-compatible with Azure Table Storage but with Cosmos's SLAs and global distribution.

Cosmos guarantees per-region single-digit-millisecond p99 reads and writes, 99.999% availability with multi-region writes, and explicit consistency level controls. Microsoft prices it via the **Request Unit (RU)** abstraction — every operation costs some number of RUs, and you provision RU/sec capacity (or use autoscale or serverless).

AWS comparison: NoSQL/Core ≈ DynamoDB (but with consistency-level knobs DynamoDB doesn't expose); Mongo API ≈ DocumentDB; Cassandra API ≈ Keyspaces.

## Partition keys and RUs

Two design decisions decide whether Cosmos DB is brilliant or unusable: the **partition key** and the **throughput model**.

Cosmos shards every container by a partition key you choose at create. A good partition key has three properties:

- **High cardinality** — many distinct values so the keyspace can split into many physical partitions.
- **Even access** — no single value is hot. A `tenantId` is fine if tenants are roughly balanced; bad if 90% of traffic is one tenant.
- **Co-locates queries** — the values that you commonly read or transact together share a key. Cross-partition queries fan out and cost more RUs.

The classic disaster pattern: picking a low-cardinality field (a status enum, a region code) as the partition key. Every write to that value lands on the same physical partition, which has a hard throughput ceiling. You hit a wall regardless of how many RUs you provision.

The **RU model** prices every operation. A point read of a 1 KB document costs ~1 RU. A query that scans 100 documents costs ~10 RUs. An indexed insert costs ~5–10 RUs. You provision throughput at the container or database level in:

- **Standard (manual) provisioned** — flat RU/sec.
- **Autoscale** — set a max RU/sec; Cosmos scales between 10% and 100% of it automatically. Slightly more expensive per RU but covers spiky workloads.
- **Serverless** — pay per request, no provisioned floor. Capped at modest throughput; good for dev/test and sporadic workloads.

Watch the **diagnostic logs** for **rate-limited (429) responses** and the **request charge** on every query — these are how you tune RU spend down once the workload is live.

## Consistency levels

Cosmos exposes five consistency levels, in order from strictest to loosest:

- **Strong** — linearisable. Reads always see the latest committed write. Highest latency; only available in single region or multi-region without multi-write.
- **Bounded staleness** — reads lag the write by at most *k* updates or *t* time. Predictable, ordered, used for global apps that need consistency budgets.
- **Session** — the **default**. A client sees its own writes immediately; other clients may lag briefly. The right choice for most user-scoped workloads.
- **Consistent prefix** — reads see writes in order but may lag. Never see a write "out of sequence."
- **Eventual** — cheapest, lowest latency. Reads may show old data and may show writes out of order for a window.

Looser consistency costs fewer RUs and gives lower latency. The picking rule: start with **Session** unless a workload requirement forces stricter; tighten only on the containers that need it.

AWS comparison: DynamoDB only exposes strongly-consistent and eventually-consistent reads. Cosmos's five-level menu is genuinely more nuanced — useful when you need it, more decisions when you don't.

## Global distribution and multi-region writes

A Cosmos account is provisioned in a **home region**; you can add **read regions** with a click — Cosmos replicates the data and serves local reads. Failover is automatic; you keep the same connection string.

Flip **multi-region writes** on and every region becomes write-capable. The trade-off:

- **Single-write region** — one region is authoritative; the others are read replicas. Conflict-free, slightly cheaper, strong/bounded-staleness available.
- **Multi-write region** — every region accepts writes; Cosmos resolves write conflicts. Default policy is **last-writer-wins** on a configurable field; alternative is a stored procedure you write. Required for sub-millisecond write latency at every region.

**Change feed** is the change-data-capture log on every container. Subscribers (a Function trigger, the Synapse Link, a custom worker) consume the feed in order and react. It's the canonical way to fan data from Cosmos out to other systems — pump it into ADLS Gen2 for analytics, trigger Search index updates, propagate to a Service Bus topic. Most production Cosmos deployments use change feed somewhere.

## Azure Table Storage — and when not to use it

**Azure Table Storage** is the original Azure NoSQL — a flat key-value store inside a storage account. Two-part key (PartitionKey + RowKey), no schema, no secondary indexes, no joins.

Table Storage is cheap and durable. It is also feature-anaemic compared to anything modern. The current advice: pick **Cosmos DB Table API** instead for any new workload. Same wire protocol, same code, but with global distribution, throughput SLA, and secondary indexes. Table Storage remains useful for *very* simple logging or metadata sidecar scenarios where Cosmos's RU cost is overkill.

## Azure Synapse Analytics

**Azure Synapse Analytics** is Microsoft's analytics platform — three compute engines plus an orchestration layer in one workspace.

- **Dedicated SQL pools** (formerly SQL Data Warehouse) — MPP columnar warehouse. Provision a fixed number of DWUs; pay for it whether queries run or not. Used for terabyte-to-petabyte structured warehouses queried by BI.
- **Serverless SQL pools** — pay per TB scanned across files in ADLS Gen2. No provisioning; you query parquet/CSV/JSON in the lake directly. Perfect for ad-hoc and data-engineering inspection.
- **Apache Spark pools** — fully managed Spark clusters; PySpark, Scala, .NET for Spark, R. Auto-scale, auto-pause. Used for transformations and ML feature engineering on the lake.
- **Pipelines** — built-in Azure Data Factory experience for orchestration.

**Synapse Link** is the headline integration — a managed change-feed pipe from Cosmos DB (or Azure SQL / Dataverse) into Synapse, with no ETL to write. You query operational data analytically without affecting the operational workload.

Microsoft's longer-term direction is **Microsoft Fabric**, which folds Synapse Spark, Data Factory, Power BI, real-time intelligence, and lakehouse storage (OneLake) into one SaaS surface. Fabric is the recommended platform for *new* analytics builds in 2026; Synapse remains supported and is the right place for existing deployments.

## Azure Data Factory

**Azure Data Factory (ADF)** is the no-code-to-low-code ETL/ELT orchestration service. The same engine sits inside Synapse Pipelines and Fabric Data Factory; standalone ADF is still what most non-Fabric analytics builds use.

Four core concepts:

- **Linked services** — connection strings to data stores and compute (a Storage account, a SQL database, a Databricks workspace, a SaaS source).
- **Datasets** — typed pointers at locations within a linked service (a specific table, a folder of CSVs).
- **Pipelines** — orchestrations of **activities** (Copy, Lookup, ForEach, Web call, Stored Procedure, Databricks notebook, Mapping Data Flow).
- **Triggers** — schedule, tumbling-window, or event-based (a blob landing in a container).

**Mapping Data Flows** are no-code, Spark-backed transformations — you draw a DAG (select, join, derive column, sink) and ADF runs it on a managed Spark cluster.

**Integration Runtimes (IR)** are where activities actually execute:

- **Azure IR** — managed compute in a region for Azure-to-Azure activities.
- **Self-hosted IR** — an agent you install on a VM in your network (on-prem or VNet) for hybrid sources.
- **Azure-SSIS IR** — runs legacy SSIS packages lifted from on-prem SQL Server.

AWS comparison: ADF ≈ AWS Glue + Step Functions + Data Pipeline. The low-code visual experience is Azure's selling point — Glue is more code-first.

## Azure Databricks

**Azure Databricks** is the Databricks lakehouse platform deployed on Azure with first-party billing through Microsoft. It is the gold standard for serious data engineering, ML, and lakehouse analytics — and a frequent alternative to Synapse Spark for shops that have made the Databricks bet.

The core abstractions:

- **Workspace** — the UI plus the management plane.
- **Clusters** — Spark clusters (interactive, job, or serverless). Autoscale; spin down when idle.
- **Delta Lake** — ACID transactions over parquet in your storage; the format every cluster reads.
- **Unity Catalog** — the central metastore, governance, lineage, and access control across workspaces and clouds. Replaces the legacy Hive metastore; Microsoft's clear direction is Unity-everywhere.
- **MLflow** — built-in experiment tracking and model registry.

Databricks vs Synapse Spark: Databricks has the longer pedigree, deeper feature set, photon engine, Unity Catalog, and Mosaic ML. Synapse Spark is integrated tightly with the rest of Synapse. For pure data engineering and ML platforms at scale, Databricks usually wins; for tight Power BI / Fabric integration, Synapse/Fabric does.

## HDInsight, Stream Analytics, and the rest

Three more analytics services you should recognise but rarely greenfield onto:

- **Azure HDInsight** — managed Hadoop / Spark / Kafka / HBase / Storm clusters on VMs. The original Azure big-data offering; largely supplanted by Databricks, Synapse, and Container-Apps-hosted Kafka clones. Still alive for workloads needing a specific Hortonworks-compatible distribution.
- **Azure Stream Analytics** — fully-managed real-time SQL over Event Hubs/IoT Hub streams. Write SQL with windowing (tumbling, hopping, sliding), join with reference data, output to Power BI, Synapse, Cosmos. Beautiful for simple stream pipelines without the operational weight of Spark Structured Streaming.
- **Microsoft Fabric Real-Time Intelligence** — the newer event-streaming surface in Fabric, built on the KQL engine. Where new streaming analytics builds will land going forward.

AWS comparison: HDInsight ≈ EMR (with the same "why am I still running this?" energy lately); Stream Analytics ≈ Kinesis Data Analytics; Real-Time Intelligence ≈ MSK + OpenSearch + Glue Streaming combined.

In [ ]:
# Provision a Cosmos DB NoSQL container with autoscale, then write & query.

RG=rg-nosql-demo
ACCT=cosmos-foundations-$RANDOM
az group create -n $RG -l eastus

# 1. Cosmos account with default Session consistency.
az cosmosdb create -n $ACCT -g $RG --kind GlobalDocumentDB \
  --default-consistency-level Session \
  --locations regionName=eastus failoverPriority=0 \
  --locations regionName=westus failoverPriority=1

# 2. Database + autoscale container; partition key /tenantId.
az cosmosdb sql database create -a $ACCT -g $RG -n app
az cosmosdb sql container create -a $ACCT -g $RG -d app -n events \
  --partition-key-path "/tenantId" \
  --max-throughput 4000

# 3. Insert via SDK (Python) — see Cosmos docs for full client setup.
python - <<'PY'
from azure.cosmos import CosmosClient
import os, json, uuid
client = CosmosClient.from_connection_string(os.environ['COSMOS_CONN'])
container = client.get_database_client('app').get_container_client('events')
container.create_item({
    'id': str(uuid.uuid4()),
    'tenantId': 'acme',
    'type': 'order.created',
    'amount': 99.0,
})
for doc in container.query_items(
    query='SELECT * FROM c WHERE c.tenantId=@t',
    parameters=[{'name':'@t','value':'acme'}],
    enable_cross_partition_query=False):
    print(doc)
PY

## Putting it together

The shape of a modern Azure data platform:

1. **Operational store** — Azure SQL (relational) and/or **Cosmos DB** (NoSQL) hold live application state.
2. **Change capture** — Cosmos's **change feed**, Azure SQL's CDC, or PostgreSQL logical decoding pumps every change into the lake.
3. **Lake** — ADLS Gen2 (notebook 05) is the durable, cheap home for raw and curated data. Delta or Iceberg tables on top give ACID.
4. **Compute** — **Azure Databricks** (lakehouse-first) or **Synapse Spark / Fabric** (Microsoft-stack-first) transforms and aggregates.
5. **Serving** — Synapse Serverless SQL or Fabric Lakehouse SQL endpoint exposes curated tables to BI; Power BI or Fabric reports query them.
6. **Orchestration** — Azure Data Factory or Synapse/Fabric Pipelines wires the schedule, dependencies, and monitoring.
7. **Streaming, if needed** — Event Hubs ingests, **Stream Analytics** or **Fabric Real-Time Intelligence** processes, output goes back into the lake or onto a Power BI dashboard.

The art is matching the engine to the access pattern. Hot, point-lookup, globally-distributed JSON → Cosmos. Big columnar warehouse queries → Synapse dedicated or Fabric Warehouse. Spark transformations → Databricks. Cheap ad-hoc inspection over the lake → Synapse Serverless SQL. Pick one engine per access pattern and the platform stays sane. Try to make one engine cover all four and you spend the next year explaining why queries take five minutes.